<h1 style="color:#ffffff; background-color:#3D3465; text-align:center; font-weight:bold; padding:16px 10px; border-radius:8px; margin-bottom:6px;">Day 1 — Train / Validation / Test Splits</h1>
<h3 style="color:#4B3F72; text-align:center; font-weight:bold; margin-top:0;">Why One Test Set Is Not Enough for Honest Tuning</h3>


<a id="toc"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">Table of Contents</h2>
<div style="border:1px solid #ccc; padding:15px 25px; border-radius:6px;">
<ol style="font-weight:bold; line-height:1.9;">
<li><a href="#section0">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</a></li>
<li><a href="#section1">1. The Problem With a Single Test Set</a></li>
<li><a href="#section2">2. The Three-Way Split</a></li>
<li><a href="#section3">3. Creating the Split in Code</a></li>
<li><a href="#section4">4. Why This Isn't Always Enough</a></li>
<li><a href="#section5">5. Common Mistakes to Avoid</a></li>
<li><a href="#section6">6. Quick Reference — Cheat Sheet</a></li>
<li><a href="#section7">7. Hands-On Lab — Building a Three-Way Split</a></li>
<li><a href="#section8">8. Best Practices &amp; Reproducibility</a></li>
<li><a href="#section9">9. Summary — What I Learned Today</a></li>
</ol>
</div>


<a id="section0"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</h2>
<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Week 3 ended with a model trained on a simple train/test split. Week 4 is about turning a model that <b>runs</b> into a model you can <b>trust</b> — and the very first crack in that trust hides inside the split itself. Today we find it, and fix it.
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Pandas version      :", pd.__version__)
print("NumPy version       :", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version      : 3.0.2
NumPy version       : 2.4.4
Scikit-learn version: 1.8.0


<a id="section1"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">1. The Problem With a Single Test Set</h2>
<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">1.1 A hidden trap</h3>
In Week 3, the data was split into <b>train</b> and <b>test</b>: train the model, then check its score once on the test set. That is honest — <i>as long as you check the test set only once</i>.

<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">1.2 What goes wrong when you tune</h3>
In practice, no one gets a model right on the first try. You try one setting, check the test score, try another setting, check again... and without meaning to, you begin <b>fitting your decisions to that specific test set</b>. The test score stops being an honest estimate of real-world performance, because information about it has leaked into your choices — even though you never technically trained on it.

In [ ]:
# A small thought experiment: imagine trying 20 random hyperparameter settings
# and picking whichever one happens to score highest on the SAME small test set.
np.random.seed(42)
fake_test_scores = np.random.normal(loc=0.75, scale=0.05, size=20)

print("20 'random' settings, scored on the same test set:")
print(np.round(fake_test_scores, 3))
print()
print(f"Best score found: {fake_test_scores.max():.3f}")
print("...but this best score is partly luck, not real improvement -- because we")
print("kept re-checking the SAME test set until one attempt looked good by chance.")

20 'random' settings, scored on the same test set:
[0.775 0.743 0.782 0.826 0.738 0.738 0.829 0.788 0.727 0.777 0.727 0.727
 0.762 0.654 0.664 0.722 0.699 0.766 0.705 0.679]

Best score found: 0.829
...but this best score is partly luck, not real improvement -- because we
kept re-checking the SAME test set until one attempt looked good by chance.


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> The more times you peek at the test set while making decisions, the less that test score means. This is exactly why a <b>separate validation set</b> exists — to give you somewhere safe to tune, while keeping the test set's honesty intact.
</div>


<a id="section2"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">2. The Three-Way Split</h2>
<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">2.1 Three sets, each with one job</h3>
The professional solution is to split the data into <b>three</b> sets instead of two, each with a single, non-overlapping job:

<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#4B3F72; color:#F2C14E;"><th style="padding:8px; text-align:left;">Set</th><th style="padding:8px; text-align:left;">Purpose</th><th style="padding:8px; text-align:left;">When It's Used</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><b>Training set</b></td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">The model learns its parameters from this</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">During <code>.fit()</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><b>Validation set</b></td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Tune choices (model, hyperparameters, features) against this</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">During development and tuning</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><b>Test set</b></td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Final, one-time, honest performance estimate</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Once, at the very end — never touched during tuning</td></tr>
</table>
</div>


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <b>The rule is strict:</b> the test set is opened exactly <b>once</b>, after every decision is final. If you tune against the test set — even accidentally, even once — you no longer have an honest estimate of how the model performs on truly new data.
</div>


<a id="section3"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">3. Creating the Split in Code</h2>
<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">3.1 Two calls to train_test_split</h3>
A three-way split is done with <b>two</b> calls to <code>train_test_split</code>: first carve off the test set, then split what remains into train and validation.

In [ ]:
from sklearn.model_selection import train_test_split

# 1) hold out 20% as the final test set
# X_temp, X_test, y_temp, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# 2) split the rest into train (75%) and validation (25%)
#    -- 75% of the remaining 80% works out to 60% of the original data
# X_train, X_val, y_train, y_val = train_test_split(
#     X_temp, y_temp, test_size=0.25, random_state=42
# )

print("We will build a real three-way split on the Titanic dataset in Section 7.")

We will build a real three-way split on the Titanic dataset in Section 7.


<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> <b>Typical proportions:</b> 60% train / 20% validation / 20% test. The exact numbers matter less than the <b>discipline</b> of keeping the test set untouched until the very end.
</div>


<a id="section4"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">4. Why This Isn't Always Enough</h2>
<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">4.1 A single validation set has its own weakness</h3>
A single validation set has its own weakness: if it happens to be an <b>unusual slice</b> of the data, your tuning decisions will be based on luck rather than signal. On smaller datasets especially, one validation split can be misleading.

In [ ]:
# Illustrating the problem: the SAME model, scored on 5 different random validation slices
# of the same data, can look meaningfully different just from the luck of the split.
np.random.seed(1)
validation_scores_across_splits = np.random.normal(loc=0.80, scale=0.04, size=5)

for i, score in enumerate(validation_scores_across_splits, start=1):
    print(f"Random validation split {i}: score = {score:.3f}")

print()
print(f"Range across splits: {validation_scores_across_splits.min():.3f} to "
      f"{validation_scores_across_splits.max():.3f}")
print("Which single split should you trust? This is exactly the problem")
print("cross-validation (Day 2) is designed to solve.")

Random validation split 1: score = 0.865
Random validation split 2: score = 0.776
Random validation split 3: score = 0.779
Random validation split 4: score = 0.757
Random validation split 5: score = 0.835

Range across splits: 0.757 to 0.865
Which single split should you trust? This is exactly the problem
cross-validation (Day 2) is designed to solve.


<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> This is exactly the problem <b>cross-validation</b>, tomorrow's topic, solves: instead of trusting one lucky-or-unlucky validation split, it averages over several, giving a far more stable estimate.
</div>


<a id="section5"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">5. Common Mistakes to Avoid</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li><b>Repeatedly checking the test set while tuning</b> — this quietly turns the test set into a second validation set, destroying its honesty.</li>
<li><b>Splitting into train/validation before splitting off the test set</b> — always carve off the test set <i>first</i>, so it never influences any earlier step.</li>
<li><b>Forgetting <code>random_state</code></b> — without it, re-running the notebook produces a different, non-reproducible split every time.</li>
<li><b>Using a validation set that is too small</b> — an unusually easy or hard slice can give a misleading picture of the model; watch for this on small datasets.</li>
<li><b>Treating a good validation score as the final answer</b> — the test score, checked once at the very end, is the only honest measure of real-world performance.</li>
</ul>
</div>


<a id="section6"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">6. Quick Reference — Cheat Sheet</h2>


<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#4B3F72; color:#F2C14E;"><th style="padding:8px; text-align:left;">Task</th><th style="padding:8px; text-align:left;">Code</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Import the tool</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from sklearn.model_selection import train_test_split</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Step 1: carve off the test set</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Step 2: split train vs. validation</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Typical proportions</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">60% train / 20% validation / 20% test</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Golden rule</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Touch the test set exactly once, after every decision is final</td></tr>
</table>
</div>


<a id="section7"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">7. Hands-On Lab — Building a Three-Way Split</h2>
<div style="border-left:5px solid #9B6FD4; background-color:#f3edfb; padding:10px 15px; margin:10px 0; border-radius:4px; color:#4d3475;">
<b>Goal:</b> Take the Week 3 Titanic dataset, build a correct 60/20/20 train/validation/test split, tune one setting using the validation set only, then check the test set exactly once at the end.
</div>


<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">7.0 Loading the dataset</h3>
We reuse <code>train_and_test2.csv</code>, the same Titanic passenger dataset from Week 3, predicting whether a passenger survived.

In [ ]:
titanic = pd.read_csv("train_and_test2.csv")
titanic = titanic.rename(columns={"2urvived": "Survived"})

features = ["Age", "Fare", "Sex", "sibsp", "Parch", "Pclass"]
X = titanic[features]
y = titanic["Survived"]

print("Shape:", X.shape)
titanic.head()

Shape: (1309, 6)


,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,Survived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">Step 1 — Create a 60/20/20 train/validation/test split with a fixed random_state</h3>


In [ ]:
from sklearn.model_selection import train_test_split

# First, carve off the final 20% test set -- and never touch it again until Step 3
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Then split the remaining 80% into 75% train / 25% validation
# (0.25 of the remaining 80% = 20% of the original data)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Training rows  : {X_train.shape[0]} ({X_train.shape[0] / len(X):.0%})")
print(f"Validation rows: {X_val.shape[0]} ({X_val.shape[0] / len(X):.0%})")
print(f"Test rows      : {X_test.shape[0]} ({X_test.shape[0] / len(X):.0%})")

Training rows  : 785 (60%)
Validation rows: 262 (20%)
Test rows      : 262 (20%)


<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">Step 2 — Train a model on the training set and tune one setting using the validation set only</h3>


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Try a few values of max_depth, checking ONLY the validation set to decide
candidate_depths = [3, 5, 10, None]
validation_results = []

for depth in candidate_depths:
    model = RandomForestClassifier(max_depth=depth, n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    val_predictions = model.predict(X_val)
    val_f1 = f1_score(y_val, val_predictions)
    validation_results.append({"max_depth": depth, "Validation F1": round(val_f1, 3)})

validation_table = pd.DataFrame(validation_results)
validation_table

,max_depth,Validation F1
0,3.0,0.396
1,5.0,0.571
2,10.0,0.500
3,NaN,0.491


<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Notice: the test set has not been touched at all yet. Every decision so far — which <code>max_depth</code> looks best — was made using the <b>validation set only</b>, exactly as the three-way split is designed for.
</div>


<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">Step 3 — Evaluate the final model on the test set exactly once and report the score</h3>


In [ ]:
best_row = validation_table.sort_values("Validation F1", ascending=False).iloc[0]
best_depth = best_row["max_depth"]
if pd.notna(best_depth):
    best_depth = int(best_depth)
else:
    best_depth = None
print(f"Best max_depth chosen using the validation set: {best_depth}")

# Retrain on train, using the chosen setting, then check the TEST set for the first and only time
final_model = RandomForestClassifier(max_depth=best_depth, n_estimators=100, random_state=42)
final_model.fit(X_train, y_train)

test_predictions = final_model.predict(X_test)
test_f1 = f1_score(y_test, test_predictions)

print(f"Final, one-time test F1-score: {test_f1:.3f}")

Best max_depth chosen using the validation set: 5
Final, one-time test F1-score: 0.513


<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> This test score is now an <b>honest</b> estimate of real-world performance, because it played no role whatsoever in choosing <code>max_depth</code>. If it had, this number could be inflated by the same kind of lucky-guess effect shown in the Section 1 thought experiment.
</div>


<h3 style="color:#F2C14E; font-weight:bold; background-color:#2D3748; display:inline-block; padding:4px 10px; border-radius:6px;">Step 4 — In Markdown, explain what would go wrong if you had tuned against the test set instead</h3>


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>If I used the test set to choose the best `max_depth`, information from
the test set would influence my model-selection decisions. The final test
score would therefore no longer be an independent estimate of performance
on unseen data, because I had already indirectly tuned the model to that
test set.
</div>


<a id="section8"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">8. Best Practices &amp; Reproducibility</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li>Always carve off the <b>test set first</b>, before doing anything else with the remaining data.</li>
<li>Fix <code>random_state=42</code> on both split calls, for a fully reproducible three-way split.</li>
<li>Make every tuning decision using the <b>validation set</b>, never the test set.</li>
<li>Check the test set <b>exactly once</b>, after every decision is already final.</li>
<li>On small datasets, remember that even a good validation score can be a matter of luck — a reason to look forward to cross-validation on Day 2.</li>
</ul>
</div>


<a id="section9"></a>
<h2 style="color:#F2C14E; background-color:#4B3F72; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">9. Summary — What I Learned Today</h2>
<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> <ul>
<li>Repeatedly checking a single test set while tuning silently <b>leaks</b> information into your decisions, making the final score dishonest.</li>
<li>The <b>three-way split</b> — train, validation, test — gives tuning a safe place to happen while keeping the test set untouched.</li>
<li>A three-way split is built with <b>two calls</b> to <code>train_test_split</code>: test set first, then train/validation from what remains.</li>
<li>The golden rule: the test set is opened <b>exactly once</b>, after every decision is final.</li>
<li>A single validation set can still mislead on smaller datasets — the exact motivation for <b>cross-validation</b>, tomorrow's topic on Day 2.</li>
</ul>
</div>
